# 03 · Quantum Machine Learning Basics

Now we connect the two worlds. **Quantum Machine Learning (QML)** uses quantum circuits as trainable models. The most common and near-term-friendly approach is the **Variational Quantum Circuit (VQC)**, also called a *parameterized quantum circuit*.

A VQC works in four stages:

1. **Encoding (state preparation):** map classical input data into qubit states.
2. **Variational layers:** apply trainable rotation gates + entangling gates.
3. **Measurement:** read out an expectation value.
4. **Classical post-processing & optimization:** compute a loss and update the parameters with a classical optimizer.

This **hybrid quantum-classical** loop is the foundation of modern QML. We'll build a working binary classifier with **PennyLane**, the leading differentiable-quantum-programming library.

In [ ]:
# Run once to install.
%pip install pennylane matplotlib scikit-learn

## 1. A toy dataset

We'll classify points from two clusters (labels `-1` and `+1`). Quantum classifiers conventionally use $\pm1$ labels because a qubit's `PauliZ` expectation value also lives in $[-1, +1]$, so the model output and the labels share the same range.

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

# Two well-separated clusters in 2D
X, y_raw = make_blobs(n_samples=100, centers=2, n_features=2,
                      cluster_std=1.0, random_state=42)

# Scale features to a small range suitable for angle encoding, and map labels to {-1, +1}
X = (X - X.mean(axis=0)) / X.std(axis=0)
Y = np.array([1 if label == 1 else -1 for label in y_raw], requires_grad=False)

plt.scatter(X[:, 0], X[:, 1], c=y_raw, cmap="coolwarm", edgecolors="k")
plt.title("Toy dataset (two classes)")
plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.show()

## 2. The quantum model (QNode)

We use **2 qubits** (one per feature). The circuit:

- **Encodes** each feature as a `RY` rotation angle (`AngleEmbedding`).
- Applies several **variational layers** of trainable rotations + `CNOT` entanglement (`StronglyEntanglingLayers`).
- **Measures** the `PauliZ` expectation of qubit 0, giving an output in $[-1, 1]$.

The `@qml.qnode` decorator turns the Python function into a differentiable quantum circuit — PennyLane can compute gradients of the output with respect to the weights, which is what makes training possible.

In [ ]:
n_qubits = 2
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def circuit(weights, x):
    # 1. Encode classical data into the qubits
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    # 2. Trainable, entangling variational layers
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    # 3. Measure -> expectation value in [-1, 1]
    return qml.expval(qml.PauliZ(0))

def variational_classifier(weights, bias, x):
    return circuit(weights, x) + bias

## 3. Loss function

We use a simple **mean-squared error** between the model output and the $\pm1$ labels. Lower loss = better fit.

In [ ]:
def square_loss(labels, predictions):
    return np.mean((labels - qml.math.stack(predictions)) ** 2)

def cost(weights, bias, X, Y):
    preds = [variational_classifier(weights, bias, x) for x in X]
    return square_loss(Y, preds)

def accuracy(labels, predictions):
    preds = np.sign(qml.math.stack(predictions))
    return np.mean(preds == labels)

## 4. Initialize parameters and train

`StronglyEntanglingLayers` needs a weight tensor of shape `(n_layers, n_qubits, 3)` — three rotation angles per qubit per layer. We initialize randomly, then run the **hybrid optimization loop**: each step, the classical optimizer (`NesterovMomentumOptimizer`) nudges the quantum circuit's parameters to reduce the loss.

In [ ]:
np.random.seed(0)
n_layers = 3
weight_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)

weights = 0.01 * np.random.randn(*weight_shape, requires_grad=True)
bias = np.array(0.0, requires_grad=True)

opt = qml.NesterovMomentumOptimizer(stepsize=0.2)
batch_size = 10

for it in range(40):
    # Mini-batch for faster, noisier updates
    batch = np.random.randint(0, len(X), (batch_size,))
    X_batch, Y_batch = X[batch], Y[batch]
    weights, bias, _, _ = opt.step(cost, weights, bias, X_batch, Y_batch)

    if (it + 1) % 5 == 0:
        preds = [variational_classifier(weights, bias, x) for x in X]
        print(f"Iter {it+1:3d} | cost: {cost(weights, bias, X, Y):.4f} "
              f"| accuracy: {accuracy(Y, preds):.2f}")

You should watch the **cost fall** and the **accuracy climb** toward ~1.0 — the quantum circuit is learning to separate the two classes.

## 5. Visualize the decision boundary

We evaluate the trained classifier on a grid to see the region it assigns to each class.

In [ ]:
xx, yy = np.meshgrid(np.linspace(-2.5, 2.5, 40), np.linspace(-2.5, 2.5, 40))
grid = np.c_[xx.ravel(), yy.ravel()]

Z = np.array([np.sign(variational_classifier(weights, bias, g)) for g in grid])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y_raw, cmap="coolwarm", edgecolors="k")
plt.title("Variational quantum classifier — decision boundary")
plt.xlabel("feature 0"); plt.ylabel("feature 1")
plt.show()

## Summary

- A **Variational Quantum Circuit** is a trainable quantum model: encode → variational layers → measure → optimize.
- Training is a **hybrid loop** — a quantum circuit computes outputs/gradients, a classical optimizer updates the parameters.
- `AngleEmbedding` loads data; `StronglyEntanglingLayers` provides trainable, entangling structure; `qml.expval(PauliZ)` reads out a prediction in $[-1, 1]$.
- This is the same template behind quantum classifiers, regressors, and quantum neural networks.

### Where to go next
- Swap in a harder, non-linearly-separable dataset (e.g. `make_moons`) and add more layers.
- Try **data re-uploading** (encode the data multiple times between layers) for more expressive models.
- Explore **quantum kernels** and `qml.kernels` for a different QML paradigm.

### References
- PennyLane — *Variational classifier* demo: https://pennylane.ai/qml/demos/tutorial_variational_classifier
- PennyLane — *Multidimensional regression with a VQC*: https://pennylane.ai/qml/demos/tutorial_qnn_multivariate_regression
- Hybrid Quantum-Classical ML with PennyLane (2025): https://arxiv.org/pdf/2511.14786